# DhakaFlow — Gate 9: FA-MAPPO arms on Colab Pro
One click: **Runtime → Run all**. Trains the three forecast-aware MAPPO arms
(`mappo_forecast`, `mappo_fp`, `mappo_full`, seed 1) with the FIXED env
(real hold action, seed pairing) and the FIXED forecaster (history primed —
forecast alive from step 1). Results sync to GCS every 10 min, so a Colab
disconnect loses at most 10 minutes. Rerun the notebook to resume the arm
list (finished arms are skipped by their DONE markers).

In [ ]:
#@title 1 · Authenticate (click the Google popup — anastasiasaat81)
from google.colab import auth
auth.authenticate_user()
!gcloud config set project direct-abacus-505920-t8 -q
!gsutil ls gs://dhakaflow-p2-direct-abacus/rl/ | head -3

In [ ]:
#@title 2 · Setup (deps + pinned upstreams + code bundle) — ~4 min
%%bash
set -e
cd /content
mkdir -p workspace && cd workspace
pip -q install "numpy<2" eclipse-sumo sumolib traci libsumo gymnasium setproctitle \
  tensorboardX absl-py easydict imageio pillow easy-torch==1.3.3 pandas scikit-learn loguru
# pinned upstream: MAPPO (official on-policy)
if [ ! -d MAPPO ]; then
  git clone -q https://github.com/marlbenchmark/on-policy.git MAPPO
  cd MAPPO && git checkout -q de66d7a4 && pip -q install -e . && cd ..
fi
# pinned upstream: BasicTS legacy + trained GWNet + dataset
if [ ! -d BasicTS_legacy ]; then
  git clone -q https://github.com/GestaltCogTeam/BasicTS.git BasicTS_legacy
  cd BasicTS_legacy && git checkout -q 4c3e2d373c647667c2123e1591f2d79c229dfcc6 && cd ..
fi
mkdir -p BasicTS_legacy/datasets/DhakaR1SensorGraph_legacy BasicTS_legacy/checkpoints
gsutil -q cp gs://dhakaflow-p2-direct-abacus/rl/legacy_dataset/adj_mx.pkl \
  gs://dhakaflow-p2-direct-abacus/rl/legacy_dataset/data.dat \
  gs://dhakaflow-p2-direct-abacus/rl/legacy_dataset/desc.json \
  BasicTS_legacy/datasets/DhakaR1SensorGraph_legacy/
gsutil -q -m cp -r "gs://dhakaflow-p2-direct-abacus/out/p3/checkpoints/GraphWaveNet" BasicTS_legacy/checkpoints/
# project code (env fixes + primed forecaster) + net/routes/context
gsutil -q cp gs://dhakaflow-p2-direct-abacus/rl/rl_code_gate9.tar.gz .
tar xzf rl_code_gate9.tar.gz
mkdir -p p2_full && cp artifacts/p2_sensor_graph_full/sensor_nodes.json p2_full/
cp artifacts/due_dso_stats/v200/DUE/003/trips_003.rou.xml.gz /tmp/due_routes.rou.xml.gz
ls BasicTS_legacy/checkpoints/GraphWaveNet/*/*/GraphWaveNet_best_val_MAE.pt
echo SETUP_OK

In [ ]:
#@title 3 · Train the three FA arms (sequential; ~hours) with 10-min GCS sync
import os, subprocess, threading, time
WS = "/content/workspace"
GCS = "gs://dhakaflow-p2-direct-abacus/rl"
os.environ["DHAKAFLOW_FC_CONTEXT"] = f"{WS}/artifacts/gates/twin_sync/forecast_context.npz"
os.environ["PYTHONPATH"] = f"{WS}:{WS}/MAPPO:{WS}/BasicTS_legacy"

def synced(arm):
    while True:
        subprocess.run(f"gsutil -q -m rsync -r {WS}/rl_runs {GCS}/runs_{arm}_s1_gate9/",
                       shell=True); time.sleep(600)

for ARM in ["mappo_forecast", "mappo_fp", "mappo_full"]:
    if subprocess.run(f"gsutil -q stat {GCS}/DONE_{ARM}_s1_gate9",
                      shell=True).returncode == 0:
        print(f"{ARM}: already DONE — skipping"); continue
    print(f"=== {ARM} ===")
    th = threading.Thread(target=synced, args=(ARM,), daemon=True); th.start()
    cmd = (f"cd {WS} && python research/control/train_sumo_signals.py "
           f"--arm {ARM} --seed 1 --algorithm_name rmappo "
           f"--experiment_name {ARM}_s1_gate9 "
           f"--n_rollout_threads 6 --n_training_threads 2 "
           f"--episode_length 660 --episode_sim_s 3600 --num_env_steps 400000 "
           f"--ppo_epoch 10 --save_interval 5 --log_interval 1 "
           f"2>&1 | tee {WS}/train_{ARM}.log")
    rc = subprocess.run(["bash", "-o", "pipefail", "-c", cmd]).returncode
    subprocess.run(f"gsutil -q -m rsync -r {WS}/rl_runs {GCS}/runs_{ARM}_s1_gate9/", shell=True)
    subprocess.run(f"gsutil -q cp {WS}/train_{ARM}.log {GCS}/train_{ARM}_s1_gate9.log", shell=True)
    marker = "DONE" if rc == 0 else "FAILED"
    subprocess.run(f"echo {marker}_rc{rc} | gsutil cp - {GCS}/{marker}_{ARM}_s1_gate9", shell=True)
    print(f"{ARM}: {marker} (rc={rc})")
print("ALL ARMS PROCESSED")